# Group analysis: ROC methods

This self-contained tutorial evaluates group-level mixed-effects models using simulations, ROC curves, and Type-I error rates. Run the cells in order; no variables from another notebook are required.

All ROC curves, sensitivity, and false-positive rates use raw, uncorrected channel p-values, not BH-adjusted q-values. At a fixed p threshold, FPR is the fraction of truly null channel tests rejected, including in all-null simulations. Separate BH-FDR experiments are supplementary multiple-testing analyses, not ROC calibration metrics.

For dataset preparation, demographics, model fitting, and the interactive statistics viewer, see [08_GroupAnalysis.ipynb](08_GroupAnalysis.ipynb).

The examples use 12 subjects and 6 Monte Carlo iterations per scenario. Increase `num_iterations` for more stable performance estimates; running all scenarios can take several minutes.

## Covariance repair and calibration

AR-IRLS now uses robust AR coefficient regression by default (`ar_estimator='robust'`), with median/MAD protection against extreme lagged predictors before bisquare refitting. BIC order selection remains OLS-based. Set `ar_estimator='ols'` in the pipeline arguments for the original MATLAB-compatible AR step. This changes first-level fits; refit all levels and do not interpret older saved study numbers as validation of the new default.

AR-IRLS first-level statistics now preserve the legacy assembled within-signal covariance blocks (not the individual-fit `cov_params()` blocks), project only the cross-signal structure to PSD, and shrink toward block independence to limit the marginal-whitened condition number to 1000. Individual-coefficient inference and within-signal contrasts are preserved relative to the legacy estimator. Large corrections and the marginal source are recorded in `stats.covariance.attrs`. This is an interim numerical approximation, not a theoretically validated variance estimator; refit first-level and group models to apply it to existing data.

For paired legacy/repaired comparisons, independent-replicate confidence intervals, coefficient coverage, all-null checks, and robust/nonrobust group comparisons, run `python -m pyBrainAnalyzIR.testing.helpers.covariance_validation --replicates 100 --subjects 12 --output covariance-roc.json`. See the [README](../README.md#covariance-roc-and-calibration-validation) for the design and limitations.

For paired ROI-before-GLM, statistics pooling, and group-before/after-ROI validation with actual BH FDR, run `python -m pyBrainAnalyzIR.testing.helpers.downstream_covariance --replicates 20 --subjects 12 --output roi-group-covariance.json`.

**AUC is not a calibration test.** Check raw-p null rejection rates and uncertainty coverage as well. All examples use the recommended `RE_formula='subject'`: one subject effect shared across conditions, channels, and chromophores. Random terms split only on explicitly named factors (`subject:cond` or `subject:cond:channel` are opt-in alternatives). Subject metadata links repeated recordings; without subject IDs the model warns and groups by recording. These examples retain `robust=False` for the group model (the first-level GLM still uses robust AR-IRLS). Historical results with the old channel-specific random effects do not certify calibration of this new model. The first-level PSD repair alone does not establish group calibration.


## ROC analysis of the group-level mixed effects model

To test the group model, we simulate datasets where the true group effects are known and use `ChannelROC` to count true and false positives over many simulated datasets.

`simdataset` takes an optional fixed-effects `formula` (same syntax as the `FE_formula` of `MixedEffects`) and a `demographics` table. Each subject's response amplitude in each channel is its row of the formula's design matrix times the term effects (`effects={term: amplitude}`; a scalar puts that amplitude on a random half of the channels, 0 means no effect, and terms not listed get 1). Continuous covariates are z-scored in the simulation, so a covariate effect is per standard deviation. `random_effect_sd` adds subject-to-subject variability.

With a formula, `simdataset` returns the truth as a table with one row per channel, chromophore (`Type`) and model term (`Condition`), named like the `MixedEffects` conditions (e.g. `Condition[HRF A]`, `Condition[HRF A]:age`).

`ChannelROC` scores every term of the analysis model against this table:
* terms that were simulated and are in the analysis model give true and false positives (sensitivity, specificity and ROC curves);
* terms in the analysis model that were not simulated (e.g. `age` in the model but not in the simulation) have no true effects, so any detection is a false positive. These only count toward the Type-I error;
* simulated terms missing from the analysis model are ignored.

`summary()` gives the sensitivity and false positive rate at p < 0.05 and the ROC area for all terms combined and for each term; `draw()` plots the same, one row per term.

In [ ]:
import warnings
import numpy as np
import pyBrainAnalyzIR.testing.simulations.simData as simData
import pyBrainAnalyzIR.testing.roc.channelROC as channelROC
import pyBrainAnalyzIR.pipelines.modules.preproccessing as prep
import pyBrainAnalyzIR.pipelines.modules.glm as glm
import pyBrainAnalyzIR.pipelines.modules.mixedeffects as mixedeffects

num_subjects = 12
num_iterations = 6
rng = np.random.default_rng(2026)
# the AR model order search often reaches its maximum on these short simulated recordings
warnings.filterwarnings('ignore', message='pmax was reached')


def group_pipeline(fe_formula):
    """First-level GLM followed by the mixed effects group model."""
    pipeline = channelROC.PipelineList([prep.intensity_opticaldensity, prep.mbll, prep.resample,
                                        glm.GLM, mixedeffects.MixedEffects])
    # robust=True adds the robust (bisquare) step, but is about 7x slower
    options = {'Fs': 2, 'noise_model': 'ar_irls', 'ar_order': 8, 'verbose': False,
               'FE_formula': fe_formula, 'RE_formula': 'subject', 'robust': False}
    return pipeline, options

Here is what a simulated dataset and its truth table look like for a model with an age covariate.

In [ ]:
demo = simData.simdemographics(num_subjects, continuous={'age': (20, 60)}, random_seed=rng)
example, truth = simData.simdataset(num_subjects, snr=0.15, formula='Beta ~ 0 + Condition + Condition:age',
                                    demographics=demo, random_effect_sd=0.3, random_seed=rng)
print(example.get_demographics().head())
truth.head(8)

### 1) Simple group analysis with two conditions

The simulation has two conditions (A and B) with no covariates. The analysis model adds an age term that was not simulated, so the `Condition:age` rows only report the Type-I error (the false positive rate of a term with no true effects).

In [ ]:
def simulate_simple(**kwargs):
    demo = simData.simdemographics(num_subjects, continuous={'age': (20, 60)}, random_seed=rng)
    return simData.simdataset(num_subjects, demographics=demo, random_seed=rng, **kwargs)


roc_simple = channelROC.ChannelROC()
roc_simple.verbose = False
roc_simple.data_simulation_function = simulate_simple
roc_simple.data_simulation_args = {'snr': 0.15, 'ncond': 2, 'random_effect_sd': 0.3,
                                   'formula': 'Beta ~ 0 + Condition'}
roc_simple.pipeline, roc_simple.pipeline_args = group_pipeline('Beta ~ 0 + Condition + Condition:age')
roc_simple.run(num_iterations)
roc_simple.summary().round(3)

In [ ]:
fig = roc_simple.draw()

### 2) Continuous covariate (age)

The response amplitude depends on age on half the channels (and the mean response is present on a random half of the channels). The analysis uses the same formula as the simulation. `MixedEffects` mean-centers numeric covariates, so `Condition[HRF A]` is the response at the mean age and `Condition[HRF A]:age` is its change with age.

In [ ]:
def simulate_age(**kwargs):
    demo = simData.simdemographics(num_subjects, continuous={'age': (20, 60)}, random_seed=rng)
    return simData.simdataset(num_subjects, demographics=demo, random_seed=rng, **kwargs)


age_formula = 'Beta ~ 0 + Condition + Condition:age'
roc_age = channelROC.ChannelROC()
roc_age.verbose = False
roc_age.data_simulation_function = simulate_age
roc_age.data_simulation_args = {'snr': 0.15, 'random_effect_sd': 0.3, 'formula': age_formula}
roc_age.pipeline, roc_age.pipeline_args = group_pipeline(age_formula)
roc_age.run(num_iterations)
roc_age.summary().round(3)

In [ ]:
fig = roc_age.draw()

### 3) Categorical group variable

Subjects are randomly split into control and patient groups. With treatment coding, `Condition[HRF A]` is the response of the control group and `Condition[HRF A]:group[T.patient]` the patient-minus-control difference, which is simulated with twice the amplitude of the control response.

In [ ]:
def simulate_groups(**kwargs):
    demo = simData.simdemographics(num_subjects, categorical={'group': ['control', 'patient']},
                                   random_seed=rng)
    return simData.simdataset(num_subjects, demographics=demo, random_seed=rng, **kwargs)


group_formula = 'Beta ~ 0 + Condition + Condition:group'
roc_group = channelROC.ChannelROC()
roc_group.verbose = False
roc_group.data_simulation_function = simulate_groups
roc_group.data_simulation_args = {'snr': 0.15, 'random_effect_sd': 0.3, 'formula': group_formula,
                                  'effects': {'Condition[HRF A]:group[T.patient]': 2}}
roc_group.pipeline, roc_group.pipeline_args = group_pipeline(group_formula)
roc_group.run(num_iterations)
roc_group.summary().round(3)

In [ ]:
fig = roc_group.draw()

`roc_table()` returns every scored test (all iterations, channels, chromophores and terms) if you want to do your own analysis, and `results(condition)` / `type_one_error(condition)` return the ROC curves and the false positive rates of a single term.